# TB CXR Diagnostic — Phase 1 & 2

EfficientNet-B0 encoder + diagnostic head for TB detection from chest X-rays.
Targets the WHO TPP triage specification: **sensitivity ≥ 90%, specificity ≥ 70%**.

**Before starting:** Runtime → Change runtime type → **GPU** (T4 recommended — 16 GB VRAM)

In [ ]:
!pip install -q timm

## 1. Dataset Setup

Upload these to your Google Drive, then mount Drive (next cell) and set paths in **Config**.

| Dataset | Kaggle slug |
|---|---|
| Shenzhen + Montgomery | `raddar/tuberculosis-tb-chest-xray-dataset` |
| TBX11K | `userhzx/tbx11k` |

**Expected Drive layout:**
```
tb_datasets/
  tb-shenzen/images/images/*.png
  tb-montgomery/images/images/*.png
  tbx11k/
    train/img/*.png   train/ann/*.json
    val/img/*.png     val/ann/*.json
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — edit paths and hyperparameters here

In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
SHENZHEN   = '/content/drive/MyDrive/tb_datasets/tb-shenzen'
MONTGOMERY = '/content/drive/MyDrive/tb_datasets/tb-montgomery'
TBX11K     = '/content/drive/MyDrive/tb_datasets/tbx11k'
OUTPUT_DIR = '/content/drive/MyDrive/tb_outputs/diagnostic'

# Source withheld entirely from training for cross-source generalisation eval.
# Change to 'shenzhen' or 'none' if you want a different held-out split.
HELD_OUT = 'montgomery'

# ── Hyperparameters ────────────────────────────────────────────────────────────
# Colab T4 has 16 GB VRAM — 512px/batch32 fits comfortably.
# On a 4 GB card use IMAGE_SIZE=320, BATCH_SIZE=16 instead.
IMAGE_SIZE      = 512
BATCH_SIZE      = 32
EPOCHS          = 50
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE        = 10   # early-stop if val AUC doesn't improve
BACKBONE        = 'efficientnet_b0'
SEED            = 42
NUM_WORKERS     = 2

## 3. One-time preprocessing

Shenzhen and Montgomery ship as very large PNGs (up to 4900×4000 px).
This resizes them once at `IMAGE_SIZE` on Drive so future epochs load fast.
**Skip this cell if you already have an `images_512/` directory in each dataset root.**

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

def _preprocess_dataset(src_dir: Path, out_dir: Path, size: int) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for src in tqdm(sorted(src_dir.glob('*.png')), desc=out_dir.parent.name):
        dst = out_dir / src.name
        if dst.exists():
            continue
        img = Image.open(src).convert('L')
        img = img.resize((size, size), Image.LANCZOS)
        img.save(dst, format='PNG', optimize=True)

for root_str, label in [(SHENZHEN, 'Shenzhen'), (MONTGOMERY, 'Montgomery')]:
    root = Path(root_str)
    src  = root / 'images' / 'images'
    dst  = root / f'images_{IMAGE_SIZE}'
    if dst.exists():
        print(f'{label}: cached at {dst}')
    elif src.exists():
        print(f'Preprocessing {label}...')
        _preprocess_dataset(src, dst, IMAGE_SIZE)
    else:
        print(f'{label}: source not found at {src}')

## 4. Imports

In [ ]:
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.multiprocessing as _mp
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm.notebook import tqdm
import timm

# Ensure DataLoader workers spawn correctly across Python versions
_mp.set_start_method('fork', force=True)

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')

## 5. Data — transforms and loaders

In [ ]:
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])

def get_val_transforms(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])


# TBX11K uses per-image Supervisely JSON tags. Maps tag → binary label; None = exclude.
_TBX11K_LABEL_MAP = {
    'healthy':          0,
    'sick_but_non-tb':  0,
    'active_tb':        1,
    'active&latent_tb': 1,
    'latent_tb':        None,
    'no_tag':           None,
}

def _img_dir(root: Path, size: int) -> Path:
    cached = root / f'images_{size}'
    return cached if cached.exists() else root / 'images' / 'images'

def load_shenzhen(root, size=512):
    return [(p, int(p.stem.rsplit('_', 1)[-1]))
            for p in sorted(_img_dir(Path(root), size).glob('*.png'))]

def load_montgomery(root, size=512):
    return [(p, int(p.stem.rsplit('_', 1)[-1]))
            for p in sorted(_img_dir(Path(root), size).glob('*.png'))]

def load_tbx11k(root, split='train', latent_as_positive=False):
    root = Path(root)
    img_dir, ann_dir = root / split / 'img', root / split / 'ann'
    label_map = dict(_TBX11K_LABEL_MAP)
    if latent_as_positive:
        label_map['latent_tb'] = 1
    samples = []
    for img_path in sorted(img_dir.glob('*.png')):
        ann_path = ann_dir / f'{img_path.name}.json'
        if not ann_path.exists():
            continue
        ann      = json.loads(ann_path.read_text())
        tag_name = ann['tags'][0]['name'] if ann['tags'] else 'no_tag'
        label    = label_map.get(tag_name)
        if label is not None:
            samples.append((img_path, label))
    return samples


class TBCXRDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
        self.labels    = [s[1] for s in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def class_counts(self):
        pos = sum(self.labels)
        return len(self.labels) - pos, pos

## 6. Model

```
Input image
     │
SharedEncoder  (EfficientNet-B0, features_only)
     │  5 feature maps at strides [2,4,8,16,32]
     ▼ features[-1]  (320 channels)
DiagnosticHead  GAP → Dropout → Linear(320,1) → logit
```

In [ ]:
class SharedEncoder(nn.Module):
    def __init__(self, backbone='efficientnet_b0', pretrained=True):
        super().__init__()
        self.net = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        self._channels = self.net.feature_info.channels()

    @property
    def out_channels(self):
        return self._channels[-1]

    def forward(self, x):
        return self.net(x)  # list of 5 feature maps


class DiagnosticHead(nn.Module):
    def __init__(self, in_channels, dropout=0.3):
        super().__init__()
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(in_channels, 1)

    def forward(self, features):
        x = self.pool(features[-1]).flatten(1)
        return self.fc(self.dropout(x))


class TBDiagnosticModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head    = head

    def forward(self, x):
        return self.head(self.encoder(x))


def build_model(backbone='efficientnet_b0', pretrained=True, dropout=0.3):
    encoder = SharedEncoder(backbone, pretrained)
    return TBDiagnosticModel(encoder, DiagnosticHead(encoder.out_channels, dropout))

## 7. Loss, metrics, and training loop

In [ ]:
def compute_pos_weight(dataset):
    neg, pos = dataset.class_counts()
    return torch.tensor([neg / pos], dtype=torch.float32)


class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight=None, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        targets = targets.float() * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits.squeeze(1), targets)


def compute_auc(labels, probs):
    if len(set(labels)) < 2:
        return float('nan')
    return roc_auc_score(labels, probs)

def _sens_spec_curve(labels, probs):
    results = []
    for thr in np.linspace(0, 1, 200):
        preds  = (probs >= thr).astype(int)
        tp = ((preds == 1) & (labels == 1)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        tn = ((preds == 0) & (labels == 0)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        results.append((thr, sens, spec))
    return results

def find_who_tpp_point(labels, probs):
    candidates = [(thr, sens, spec)
                  for thr, sens, spec in _sens_spec_curve(labels, probs)
                  if sens >= 0.90]
    if not candidates:
        return None
    thr, sens, spec = max(candidates, key=lambda x: x[2])
    return {'threshold': thr, 'sensitivity': sens, 'specificity': spec,
            'meets_minimum_tpp': spec >= 0.70, 'meets_optimal_tpp': spec >= 0.80}

def evaluate(labels, probs):
    labels, probs = np.array(labels), np.array(probs)
    return {'auc': compute_auc(labels, probs), 'who_tpp': find_who_tpp_point(labels, probs)}

def print_report(metrics):
    print(f"  AUC : {metrics['auc']:.4f}")
    tpp = metrics['who_tpp']
    if tpp:
        thr, sens, spec = tpp['threshold'], tpp['sensitivity'], tpp['specificity']
        min_ok = '\u2713' if tpp['meets_minimum_tpp'] else '\u2717'
        opt_ok = '\u2713' if tpp['meets_optimal_tpp'] else '\u2717'
        print(f'  WHO TPP (sens\u226590%): thr={thr:.3f}  sens={sens:.3f}  spec={spec:.3f}  '
              f'min(spec\u226570%)={min_ok}  opt(spec\u226580%)={opt_ok}')
    else:
        print('  WHO TPP: no threshold reaches sensitivity >= 90%')


class Trainer:
    def __init__(self, model, optimizer, criterion, scheduler, device, output_dir):
        self.model      = model
        self.optimizer  = optimizer
        self.criterion  = criterion
        self.scheduler  = scheduler
        self.device     = device
        self.output_dir = Path(output_dir)
        self.scaler     = torch.amp.GradScaler(device.type)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _run_epoch(self, loader, train):
        self.model.train(train)
        total_loss, all_probs, all_labels = 0.0, [], []

        with torch.set_grad_enabled(train):
            for imgs, labels in tqdm(loader, desc='train' if train else 'val', leave=False):
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                with torch.amp.autocast(self.device.type):
                    logits = self.model(imgs)
                    loss   = self.criterion(logits, labels)
                if train:
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
                total_loss += loss.item() * len(labels)
                probs = torch.sigmoid(logits).squeeze(1).float().detach().cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.cpu().numpy())

        n = len(loader.dataset)
        return total_loss / n, np.concatenate(all_probs), np.concatenate(all_labels)

    def fit(self, train_loader, val_loader, epochs, patience=10):
        best_auc, no_improve = 0.0, 0
        for epoch in range(1, epochs + 1):
            tr_loss, _, _              = self._run_epoch(train_loader, train=True)
            val_loss, val_probs, val_y = self._run_epoch(val_loader,   train=False)
            self.scheduler.step()
            metrics = evaluate(val_y, val_probs)
            print(f'\nEpoch {epoch:3d}  train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}')
            print_report(metrics)
            if metrics['auc'] > best_auc:
                best_auc, no_improve = metrics['auc'], 0
                self._save_checkpoint(epoch, metrics)
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'\nEarly stop at epoch {epoch} (no AUC gain for {patience} epochs).')
                    break
        print(f'\nBest val AUC: {best_auc:.4f}')

    def evaluate_loader(self, loader):
        _, probs, labels = self._run_epoch(loader, train=False)
        return probs, labels

    def load_best(self):
        # weights_only=False needed: checkpoint includes optimizer state + metrics
        ckpt  = torch.load(self.output_dir / 'best_model.pt', map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt['model'])
        epoch = ckpt['epoch']
        auc   = ckpt['metrics']['auc']
        print(f'Loaded best model — epoch {epoch}  AUC={auc:.4f}')

    def _save_checkpoint(self, epoch, metrics):
        torch.save({'epoch': epoch, 'model': self.model.state_dict(),
                    'optimizer': self.optimizer.state_dict(), 'metrics': metrics},
                   self.output_dir / 'best_model.pt')
        auc = metrics['auc']
        print(f'  -> checkpoint saved (AUC={auc:.4f})')

## 8. Run training

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_splits(held_out):
    rng = np.random.default_rng(42)
    train_s, val_s, held_s = [], [], []
    for name, path, loader in [('shenzhen',   SHENZHEN,   load_shenzhen),
                                ('montgomery', MONTGOMERY, load_montgomery)]:
        if not path:
            continue
        samples = loader(path, size=IMAGE_SIZE)
        if name == held_out:
            held_s.extend(samples)
        else:
            idx   = rng.permutation(len(samples))
            n_val = max(1, int(len(samples) * 0.15))
            val_s.extend(  [samples[i] for i in idx[:n_val]])
            train_s.extend([samples[i] for i in idx[n_val:]])
    if TBX11K:
        for split, target in [('train', train_s), ('val', val_s)]:
            target.extend(load_tbx11k(TBX11K, split=split))
    return train_s, val_s, held_s

def make_loader(samples, transform, weighted=False):
    ds = TBCXRDataset(samples, transform)
    sampler = None
    if weighted:
        lbls    = np.array(ds.labels)
        counts  = np.bincount(lbls)
        weights = (1.0 / counts)[lbls]
        sampler = WeightedRandomSampler(weights, len(weights))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                        shuffle=(sampler is None), num_workers=NUM_WORKERS,
                        pin_memory=True)
    return loader, ds


# ── Train ──────────────────────────────────────────────────────────────────────
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_s, val_s, held_s = build_splits(HELD_OUT)
print(f'Samples — train: {len(train_s)}  val: {len(val_s)}  held-out: {len(held_s)}')

train_loader, train_ds = make_loader(train_s, get_train_transforms(IMAGE_SIZE), weighted=True)
val_loader,   _        = make_loader(val_s,   get_val_transforms(IMAGE_SIZE))

model      = build_model(BACKBONE, pretrained=True, dropout=0.3).to(device)
pos_weight = compute_pos_weight(train_ds).to(device)
criterion  = LabelSmoothBCE(pos_weight=pos_weight, smoothing=LABEL_SMOOTHING)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(model, optimizer, criterion, scheduler, device, OUTPUT_DIR)
trainer.fit(train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE)

# ── Cross-source generalisation evaluation ─────────────────────────────────────
if held_s and HELD_OUT != 'none':
    print(f'\n--- Held-out source [{HELD_OUT}] evaluation ---')
    trainer.load_best()
    held_loader, _ = make_loader(held_s, get_val_transforms(IMAGE_SIZE))
    probs, labels  = trainer.evaluate_loader(held_loader)
    print_report(evaluate(labels, probs))